# EfficientNetB0

This notebook explains how to train and evaluate the EfficientNet-B0 model on the FLIM datasets: eggs, larvae and cysts.

EfficientNet is a family of convolutional neural networks introduced by Mingxing Tan and Quoc Le (2019). The main idea is compound scaling,
which scales network depth, width and input resolution together. Compound scaling improves accuracy for a given computational budget.
EfficientNet-B0 is the base configuration and provides a practical balance between accuracy and computational cost.

Paper: Tan, M. & Le, Q. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. https://arxiv.org/abs/1905.11946

PyTorch implementation: this notebook uses the `torchvision.models` implementation named `efficientnet_b0`. The implementation supports loading
ImageNet pretrained weights and adapting the final classification layer for our tasks. See the PyTorch documentation for details: https://docs.pytorch.org/vision/main/models/efficientnet.html

What you can do in this notebook: train from scratch with random initialization or fine-tune from ImageNet pretrained weights. Configure dataset,
training schedule and other hyperparameters in the `CONFIG_*` blocks.

Quick start:
1. Activate the virtual environment: `source venv/bin/activate`
2. Install dependencies: `pip install -r requirements.txt`
3. Run the cells in order. Edit the `CONFIG_*` blocks to set dataset, data fraction, number of epochs and other options.

Notes:
Paths in this notebook are configured relative to the repository root using `NOTEBOOK_DIR`, `BASELINE_DIR` and `PROJECT_ROOT`.
Results, model checkpoints and JSON reports are written to folders such as `efficientnetb0/efficientnetb0_scratch/eggs` or `efficientnetb0/efficientnetb0_pretrained/eggs` depending on the chosen mode.


In [1]:
# Imports and notebook setup
import sys
from pathlib import Path
import json

# Core PyTorch stack
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import transforms
from torchinfo import summary

# Data handling and visualization
import os
import seaborn as sns
import PIL
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import cohen_kappa_score

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# Repository paths
NOTEBOOK_DIR = Path.cwd().resolve()
BASELINE_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = BASELINE_DIR.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(BASELINE_DIR))

# Project modules
from config import get_dataset_paths, get_split_path_incremental
from src.dataset import DataModuleParasite
from src import utils
from src import models
from src import trainer

## Egg Dataset

In [2]:
# Eggs dataset configuration
CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': [1, 2, 3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100]
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_EGG
dataset_name = config['dataset_name']
type_model = 'efficientnetb0'
image_size = config['image_size']

transforms_egg = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_egg)


### Training from Scratch

This section trains EfficientNet-B0 on the eggs dataset using random initialization.

In [9]:
# Training mode specific settings
model_name = 'efficientnetb0_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

Model: efficientnetb0_scratch | Dataset: eggs
FLOPs: 365448288.0
Parameters: 4019077.0


In [10]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping triggered at epoch 23 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 22 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 21 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 25 (split 1, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 21 (split 2, percentage 5%) - No improvement for 20 epochs


...

Early stopping triggered at epoch 93 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 91 (split 3, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 49 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 49 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 46 (split 3, percentage 100%) - No improvement for 20 epochs


In [11]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_scratch/eggs/efficientnetb0_scratch_aggregated_classification_report_eggs.txt


### Fine-Tuning from ImageNet

This section fine-tunes EfficientNet-B0 on the eggs dataset using ImageNet-pretrained weights.

In [12]:
# Fine-tune model
model_name = 'efficientnetb0_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [13]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping triggered at epoch 89 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 28 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 62 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 55 (split 1, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 61 (split 2, percentage 5%) - No improvement for 20 epochs


...

Early stopping triggered at epoch 82 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 61 (split 3, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 40 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 34 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 44 (split 3, percentage 100%) - No improvement for 20 epochs


In [14]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_pretrained/eggs/efficientnetb0_pretrained_aggregated_classification_report_eggs.txt


## Cyst Dataset

In [15]:
# Cyst dataset configuration
CONFIG_CYST = {
    'dataset_name': 'cysts',
    'split': [1, 2, 3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100],
    'num_classes': 7,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_CYST
dataset_name = config['dataset_name']
type_model = 'efficientnetb0'
image_size = config['image_size']

transforms_cyst = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_cyst)


### Training from Scratch

This section trains EfficientNet-B0 on the cyts dataset using random initialization.

In [16]:
# Training mode specific settings
model_name = 'efficientnetb0_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

Model: efficientnetb0_scratch | Dataset: cysts
FLOPs: 365445728.0
Parameters: 4016515.0


In [17]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping triggered at epoch 21 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 21 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 28 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 72 (split 1, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 56 (split 2, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 81 (split 3, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 66 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 99 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 77 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 38 (split 3, percentage 100%) - No improvement for 20 epochs


In [18]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_scratch/cysts/efficientnetb0_scratch_aggregated_classification_report_cysts.txt


### Fine-Tuning from ImageNet

This section fine-tunes EfficientNet-B0 on the cysts dataset using ImageNet-pretrained weights.

In [19]:
# Fine-tune model
model_name = 'efficientnetb0_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [20]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping triggered at epoch 41 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 33 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 98 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 79 (split 2, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 91 (split 3, percentage 5%) - No improvement for 20 epochs


...

Early stopping triggered at epoch 80 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 93 (split 3, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 46 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 49 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 36 (split 3, percentage 100%) - No improvement for 20 epochs


In [21]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_pretrained/cysts/efficientnetb0_pretrained_aggregated_classification_report_cysts.txt


## Larvae Dataset

In [22]:
# Larvae dataset configuration
CONFIG_LARVAE = {
    'dataset_name': 'larvae',
    'split': [1,2,3],
    'percentage': [1, 5, 50, 100], #[1, 5, 25, 50, 75, 100],
    'num_classes': 2,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 100,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Setup for training 
config = CONFIG_LARVAE
dataset_name = config['dataset_name']
type_model = 'efficientnetb0'
image_size = config['image_size']

transforms_larvae = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataloaders = utils.create_dataloaders(config=config, transforms=transforms_larvae)


### Training from Scratch

This section trains EfficientNet-B0 on the larvae dataset using random initialization.

In [23]:
# Training mode specific settings
model_name = 'efficientnetb0_scratch'
pre_trained = False
path = f"{type_model}/{model_name}/{dataset_name}"

# Create the model
model_scratch, _, _, _ = models.create_model(
    type_model=type_model,
    description_path=f"{path}/{model_name}_model_params_{dataset_name}.txt",
    config=config,
    pre_trained=pre_trained
)

# Model complexity summary
input_size = (1, 3, config['image_size'], config['image_size'])
n_flops, n_params = utils.count_flops(model_scratch, input_size=input_size)
print(f"Model: {model_name} | Dataset: {dataset_name}")
print(f"FLOPs: {n_flops}")
print(f"Parameters: {n_params}")

Model: efficientnetb0_scratch | Dataset: larvae
FLOPs: 365439328.0
Parameters: 4010110.0


In [24]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)

Early stopping triggered at epoch 26 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 21 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 25 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 81 (split 1, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 52 (split 2, percentage 5%) - No improvement for 20 epochs


...

Early stopping triggered at epoch 50 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 67 (split 3, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 30 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 27 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 30 (split 3, percentage 100%) - No improvement for 20 epochs


In [25]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_scratch/larvae/efficientnetb0_scratch_aggregated_classification_report_larvae.txt


### Fine-Tuning from ImageNet

This section fine-tunes EfficientNet-B0 on the larvae dataset using ImageNet-pretrained weights.

In [26]:
# Fine-tune model
model_name = 'efficientnetb0_pretrained'
pre_trained = True
path = f"{type_model}/{model_name}/{dataset_name}"

In [27]:
# Training loop
# This block trains the model and stores the training history for later analysis.
historic_train = trainer.train_loop(
    config, dataloaders, path, dataset_name, 
    model_name=model_name, type_model=type_model, 
    pre_trained=pre_trained)

# Save the training history
with open(f'{path}/{model_name}_historic_{config["dataset_name"]}.json', 'w') as f:
    json.dump(historic_train, f)
    

Early stopping triggered at epoch 24 (split 1, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 22 (split 2, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 54 (split 3, percentage 1%) - No improvement for 20 epochs


Early stopping triggered at epoch 61 (split 1, percentage 5%) - No improvement for 20 epochs


Early stopping triggered at epoch 30 (split 2, percentage 5%) - No improvement for 20 epochs


...

Early stopping triggered at epoch 34 (split 2, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 51 (split 3, percentage 50%) - No improvement for 20 epochs


Early stopping triggered at epoch 35 (split 1, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 30 (split 2, percentage 100%) - No improvement for 20 epochs


Early stopping triggered at epoch 42 (split 3, percentage 100%) - No improvement for 20 epochs


In [28]:
# Training curves
utils.loss_and_accuracy_split(config, path, model_name, show_plot=False)
utils.loss_and_accuracy_aggregate(config, path, model_name, show_plot=False)

# Test set evaluation
results_test = utils.evaluate_test_set(
    config,
    path,
    model_name,
    dataloaders,
    type_model=type_model,
    pre_trained=pre_trained
)

# Save test metrics
with open(f'{path}/{model_name}_test_results_{config["dataset_name"]}.json', 'w') as f:
    json.dump(results_test, f)

# Save report files (confusion matrix, classification report, etc.)
utils.calculate_and_save_reports(config, path, model_name)

Aggregated classification reports saved to efficientnetb0/efficientnetb0_pretrained/larvae/efficientnetb0_pretrained_aggregated_classification_report_larvae.txt
